# Workshop 1: Checking Your Foundations: Core Concepts Self-Assessment

**Course:** Geoprocesamiento  
**Program:** Maestría en Geomática  
**Academic Term:** 2026-2  
**Lecturer:** Liliana Castillo Villamor

**Student:** Luis Gabriel Bautista Montealegre 


---
Throughout this notebook, all demonstrative calculations, cloud-masking routines, and annual anomaly maps are executed using San Luis de Gaceno (my hometown) as the reference study area. Find a geopackage of Colombian Municipalities [**in this Drive Folder**](https://drive.google.com/drive/folders/1LBIwi4xA8khQqD8JimTCjIs2VxbYfr0b?usp=sharing) 

Your Personal Study Area: You are expected to replicate, adapt, and execute this entire workflow for your own chosen region of interest.

Interactive Challenges: As you progress through the notebook, you will encounter dedicated Challenge sections. These require you to modify the code parameters, adapt spatial boundaries, compute custom metrics, and interpret your specific results.

Final Submission & GitHub Repository: You must publish your completed notebook to a dedicated public GitHub repository created specifically for this Geoprocessing course. Maintain a single, structured repository for all course assignments. Upon completion, submit the direct URL of your repository for assessment.

---
## Learning Objectives
By completing this self-assessment and proceeding through the workshop, you will ensure you can:
1. Evaluate Your Remote Sensing Readiness prior to executing automated workflows.
2. Evaluate your ability to distinguish between vector manipulation (`GeoPandas`) and raster array processing (`Rasterio`), identifying when to use each library within a spatial workflow.
3. Benchmark your knowledge of maps algebra, temporal compositing, and pixel-level quality control (cloud masking).
4. Recognise specific technical areas requiring theoretical revision before completing the practical coding challenges in this course.

## 0. Library Imports
Importing the required libraries 
> 🎯**Your Turn 1:** Add comments to explain what each library does.


In [1]:
import geopandas as gpd #georeferenced vector data
import rasterio #working with georeferenced rasters
import rasterio.mask #cropping rasters
import rasterio.warp #reprojection and spatial transformation
from rasterio.enums import Resampling #changing a raster's resolution
import ee #Google Earth Engine API
import numpy as np #working with matrices and numerical arrays
import matplotlib.pyplot as plt #visualizing graphs and maps
from pathlib import Path #managing files and folders

Eestablish an authenticated session with the Google Earth Engine (GEE) Python API. 

The standard initialization call `ee.Initialize()` verifies your GEE credentials. Wrapping this step inside a `try-except` block ensures that any authentication errors or connection failures are caught gracefully rather than crashing your notebook execution.

In [3]:
ee.Authenticate()

Enter verification code:  4/1ATsMZqBXgmAxe-FsEKd94I51JD11ZL8Ue5dOZsDEpkqFMYS3FIy2k1dgMvM



Successfully saved authorization token.


In [4]:
# Initialise Google Earth Engine API
try:
    ee.Initialize() # Establish a connection to Google Earth Engine
    print("Google Earth Engine initialised successfully.") # Confirmation message if the connection is successful
except Exception as e:
    print(f"Error initialising GEE: {e}") # Displays an error message on the screen to identify the problem

Google Earth Engine initialised successfully.


## 1: Vector Alignment & Preprocessing

1.  Setting up a dynamic path using Python’s `pathlib.Path` module to point towards your spatial datasets.
2.  Inspecting the original spatial reference of the dataset and reprojecting it to EPSG:9377 (Magna-Sirgas / Origen Nacional)

In [5]:
root_folder=Path(r"C:\Users\Luis G Bautista M\Documents\Doctorado\Geoprocesamiento\Taller1")
# Load Colombian municipalities
gdf = gpd.read_file(root_folder /"municipios_colombia.gpkg")

# Display initial spatial reference system
print(f"Initial CRS: {gdf.crs}")

# Reproject to EPSG:9377
gdf_unico = gdf.to_crs(epsg=9377)
#Check the attributes table of our geopandas dataframe
gdf_unico.head()

Initial CRS: EPSG:3116


,DPTO_CCDGO,MPIO_CCDGO,MPIO_CNMBR,MPIO_CDPMP,VERSION,AREA,LATITUD,LONGITUD,STCTNENCUE,STP3_1_SI,...,STP51_PRIM,STP51_SECU,STP51_SUPE,STP51_POST,STP51_13_E,STP51_99_E,Shape_Leng,Shape_Area,Codigo_Mun,geometry
0,18,001,FLORENCIA,18001,2018,2.547638e+09,1.749139,-75.558239,71877.0,32.0,...,48848.0,59610.0,21898.0,4592.0,5892.0,3799.0,2.942508,0.206928,18001,"MULTIPOLYGON (((4730856.146 1800689.038, 47308..."
1,18,029,ALBANIA,18029,2018,4.141221e+08,1.227865,-75.882327,2825.0,24.0,...,1940.0,1712.0,231.0,41.0,215.0,46.0,1.112829,0.033618,18029,"MULTIPOLYGON (((4677933.827 1709133.846, 46779..."
2,18,094,BELÉN DE LOS ANDAQUÍES,18094,2018,1.191619e+09,1.500923,-75.875645,4243.0,54.0,...,3541.0,3340.0,490.0,119.0,720.0,123.0,2.234657,0.096745,18094,"MULTIPOLYGON (((4690015.614 1751610.86, 469000..."
3,18,247,EL DONCELLO,18247,2018,1.106076e+09,1.791386,-75.193944,8809.0,0.0,...,7571.0,6287.0,1029.0,228.0,1095.0,171.0,3.154370,0.089867,18247,"MULTIPOLYGON (((4737450.122 1814755.048, 47374..."
4,18,256,EL PAUJÍL,18256,2018,1.234734e+09,1.617746,-75.234043,5795.0,0.0,...,6072.0,4066.0,639.0,108.0,916.0,99.0,3.529316,0.100309,18256,"MULTIPOLYGON (((4736905.653 1802381.382, 47376..."


> 🎯**Your Turn 2:** Create a new column in your GeoDataFrame named `area_km2` to calculate the surface area of each municipality in square kilometres ($\text{km}^2$). 


In [7]:
# Convert square meters (m²) to square kilometers (km²)
gdf["AREA_KM2"] = gdf["AREA"] / 1_000_000

# Show municipality and area in km²
print(gdf[["MPIO_CNMBR", "AREA_KM2"]])

                  MPIO_CNMBR     AREA_KM2
0                  FLORENCIA  2547.637532
1                    ALBANIA   414.122070
2     BELÉN DE LOS ANDAQUÍES  1191.618572
3                EL DONCELLO  1106.076151
4                  EL PAUJÍL  1234.734145
...                      ...          ...
1117              FUSAGASUGÁ   193.952877
1118     SAN JUAN DE RIOSECO   314.087274
1119                   HONDA   304.886912
1120                SABANETA    15.835319
1121             LA ESTRELLA    36.631794

[1122 rows x 2 columns]


In [6]:
# Select Manizales
la_palma = gdf_unico[
    gdf_unico["MPIO_CNMBR"] == "LA PALMA"
].copy()

# Calculate area in square kilometers
area_la_palma_km2 = la_palma.geometry.area.iloc[0] / 1_000_000

print(f"Area of La Palma: {area_la_palma_km2:.2f} km²")

Area of La Palma: 190.59 km²


❓ Reflection Check 1: What is a GeoDataFrame, and how does it differ from a standard Pandas DataFrame?

R=/ A GeoDataFrame is a Pandas DataFrame enhanced with spatial geometry and coordinate reference system information, allowing both tabular and geospatial analysis.

> 🎯**Your Turn 3:** Filter the national vector layer to extract your chosen municipality of interest. Store your selection in a new spatial variable (e.g., gdf_muni).

In [7]:
# TODO: Complete the code to filter a specific polygon and apply a spatial buffer.

# 1. Filter the GeoDataFrame for the borough 'Camden'
# Filter the GeoDataFrame for the municipality of MANIZALES
gdf_muni = gdf_unico[gdf_unico["MPIO_CNMBR"] == "LA PALMA"].copy()

# Assign the selected municipality to the variable used in the buffer exercise
selected_borough = gdf_muni.copy()

# Verify the selected municipality
print(f"Selected Municipality: {selected_borough['MPIO_CNMBR'].iloc[0]}")

Selected Municipality: LA PALMA


> 🎯**Your Turn 4:** Complete the missing parameters (___) in the code block below to apply a 500-metre buffer around your selected municipality's boundary and assign it as the active spatial geometry.

In [8]:
# 2. Apply a 500-metre buffer to the selected geometry
# HINT: Call the .buffer() method on the geometry column
buffered_shape = selected_borough.geometry.buffer(500) # 500 indicates the buffer distance

# 3. Create a new GeoDataFrame containing the buffered result
gdf_buffered = selected_borough.copy()
gdf_buffered.set_geometry(buffered_shape, inplace=True) #buffered_shape stores the new geometry extended by 500 meters

# Output geometry confirmation
print(f"Buffered Geometry Type: {gdf_buffered.geometry.type.iloc[0]}")

Buffered Geometry Type: Polygon


## 2. Earth Engine Extraction & Local Raster Resampling
Exctract Sentinel-2 surface reflectance imagery via GEE, derive the Normalised Difference Vegetation Index ($NDVI$), and resample the grid cell resolution from 10 metres to Landsat imagery using bilinear interpolation.

### Cloud Masking using Sentinel-2 Scene Classification (SCL)
Define a custom Python function (mask_s2_clouds_scl) designed to process Sentinel-2 Level-2A surface reflectance data in Earth Engine. Itinspects the Scene Classification Layer (SCL) band to identify problematic observations. It isolates pixel values representing cloud shadows (class 3), medium-probability clouds (class 8), high-probability clouds (class 9), and cirrus clouds (class 10), combining them into a single binary mask. By applying .updateMask(), all identified cloudy and shadow pixels are masked out (set to transparent/invalid), ensuring that subsequent temporal compositing routines rely strictly on clear surface reflectances.

In [9]:
def mask_s2_clouds_scl(image):
    """Masks clouds, cloud shadows, and cirrus using the SCL band."""
    scl = image.select('SCL')
    
    # Identify unwanted pixel classes
    cloud_shadows = scl.eq(3) # Identify pixels corresponding to cloud shadows
    clouds_medium = scl.eq(8) # Identify pixels with medium-probability clouds
    clouds_high = scl.eq(9)   # Identify pixels with high-probability clouds
    cirrus = scl.eq(10)       # Identify pixels corresponding to cirrus (high, thin, wispy clouds)
    
    # Combine all mask conditions (1 = invalid pixel)
    mask = cloud_shadows.Or(clouds_medium).Or(clouds_high).Or(cirrus).Not()
    
    # Update image mask and retain properties
    return image.updateMask(mask)

## 3. Extracting, Masking, and Computing NDVI over the Area of Interest

This section converts our local vector boundary into a native Earth Engine geometry (`ee.Geometry`) to query server-side satellite collections. 

The workflow performs four main operations:
1. Reprojects the local bounding box and geometry.
2. Queries the `COPERNICUS/S2_SR_HARMONIZED` collection over the specified date range and applies a preliminary metadata cloud filter (`CLOUDY_PIXEL_PERCENTAGE < 30`).
3. Maps the custom `mask_s2_clouds_scl` function across every image in the collection to remove remaining clouds and shadows.
4. Calculates a temporal median composite, computes the Normalized Difference Vegetation Index (NDVI) using Near-Infrared ($\text{B8}$) and Red ($\text{B4}$) bands:

$$\text{NDVI} = \frac{\text{B8} - \text{B4}}{\text{B8} + \text{B4}}$$

Finally, the resulting index is clipped directly to the exact municipal spatial boundary (`ee_muni_geom`).



In [10]:
# Convert vector bounding box to EPSG:4326 
bbox = gdf_muni.to_crs(epsg=4326).total_bounds
ee_bounds = ee.Geometry.BBox(bbox[0], bbox[1], bbox[2], bbox[3])

geojson_geom = gdf_muni.to_crs(epsg=4326).geometry.iloc[0].__geo_interface__
ee_muni_geom = ee.Geometry(geojson_geom)

# Filter Sentinel-2 collection
s2_collection = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(ee_bounds)
    .filterDate("2023-01-01", "2023-12-31") 
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 30))
    .map(mask_s2_clouds_scl)
)

# Compute median mosaic, calculate NDVI, and CLIP to exact municipal boundary
median_image = s2_collection.median()
ndvi_image = median_image.normalizedDifference(["B8", "B4"]).rename("NDVI").clip(ee_muni_geom)

print("Sentinel-2 NDVI layer clipped to municipality boundary successfully.")

Sentinel-2 NDVI layer clipped to municipality boundary successfully.


> 🎯 **Your Turn 5**: Why is it necessary to reproject our local vector boundary to EPSG:4326 before querying Earth Engine, even though we previously converted our layer to EPSG:9377 for area calculations?
>
> R=/ EPSG:9377 is a projected CRS (Coordinate Reference System) used locally for accurate area and distance calculations in metres. We reproject the local vector boundary to EPSG:4326 because Earth Engine uses geographic coordinates (longitude/latitude) to define and query geometries.

> 🎯 **Your Turn 6:** How can you inspect all available band names within a single Sentinel-2 image or collection using Earth Engine? Write the command required to print this list to the console.

In [11]:
print(s2_collection.first().bandNames().getInfo()) # Display the names of all the bands in the first image of the collection
#B2, B3, B4 = visible bands (blue, green, red)
#B8: near-infrared (NIR)
#B11, B12: short-wave infrared (SWIR)
#SCL: scene classification, used to identify clouds, shadows, and cirrus clouds
#QA60: quality information for clouds

['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12', 'AOT', 'WVP', 'SCL', 'TCI_R', 'TCI_G', 'TCI_B', 'MSK_CLDPRB', 'MSK_SNWPRB', 'QA10', 'QA20', 'QA60', 'MSK_CLASSI_OPAQUE', 'MSK_CLASSI_CIRRUS', 'MSK_CLASSI_SNOW_ICE']


#### 3.1 Interactive Spatial Visualisation using Folium

This section renders our server-side Earth Engine NDVI composite on a client-side interactive map using `folium`.

The rendering process consists of four main steps:
1. Setting minimum (`0.0`) and maximum (`0.8`) spectral thresholds to highlight vegetation density gradients.
2. Retrieving web map tile URLs from Google Earth Engine using `.getMapId()`.
3. Extracting the exact geometric centroid of chosen municipality  to automatically position the map viewport.
4. Adding the GEE raster stream and layer toggles directly into the interactive canvas.


In [12]:
import folium
# Set visualization parameters for the NDVI layer (-1.0 to 1.0 scale)
ndvi_vis = {
    'min': 0.0,
    'max': 0.8,
    'palette': ['blue', 'white', 'green']  # Represents water/bare soil through to dense vegetation
}

# Fetch the map URL dictionary directly from Google Earth Engine servers
map_id_dict = ndvi_image.getMapId(ndvi_vis)

# Calculate the precise centroid in the projected CRS (metres) before reprojecting to WGS84 (EPSG:4326)
centroid_point = gdf_muni.geometry.centroid.to_crs(epsg=4326).iloc[0]
centroid = [centroid_point.y, centroid_point.x]

# Initialise interactive Folium map centred on the municipal boundary
m = folium.Map(location=centroid, zoom_start=11)

# Add the GEE Sentinel-2 NDVI raster layer to the map interface
folium.TileLayer(
    tiles=map_id_dict['tile_fetcher'].url_format,
    attr='Google Earth Engine',
    name='Sentinel-2 NDVI',
    overlay=True,
    control=True
).add_to(m)

# Render layer contal visualizar la imagen queda solo la mitad de la imagen dentro derols and display the interactive map
folium.LayerControl().add_to(m)
m

> 🎯 **Your Turn 7:** Modify the visualization parameters in the code block above to change the color palette and scale threshold:
> - Adjust the maximum NDVI value (`max`)** to **0.7.
> - Change the color palette so that values transition towards red for high vegetation density.
> 


In [13]:
# Low NDVI in red and high NDVI in green
ndvi_vis2 = {
    "min": 0,
    "max": 0.7,
    "palette": ["red", "yellow", "green"]
}
print(ndvi_vis2) #Check value settings and palette

#Print map with changes
map_id_dict = ndvi_image.getMapId(ndvi_vis2)

# Create a new map
new_ndvi = folium.Map(location=centroid, zoom_start=11)

folium.TileLayer(
    tiles=map_id_dict['tile_fetcher'].url_format,
    attr='Google Earth Engine',
    name='Sentinel-2 NDVI',
    overlay=True,
    control=True
).add_to(new_ndvi)

folium.LayerControl().add_to(new_ndvi)

new_ndvi

{'min': 0, 'max': 0.7, 'palette': ['red', 'yellow', 'green']}


**NDVI MAP:** The map shows a predominance of high values represented by green colors, indicating dense and vigorous vegetation across most of the study area. Scattered yellow areas are observed, representing zones with moderate vegetation vigor, associated with lower density or transitions in land cover. On the other hand, low values, represented by the color red, are limited, suggesting sparse vegetation, which is associated with the town center of the selected municipality.

## 4. Cross-Sensor Comparison & Spatial Resampling: Landsat 8 OLI vs Sentinel-2 MSI

Compare vegetation index metrics derived from two distinct satellite constellations over your chosen municipality: **Landsat 8 OLI**  and **Sentinel-2 MSI** 

This multi-sensor workflow involves three core steps:
1. Radiometric scaling of surface reflectance bands and calculation of 30-metre NDVI.
2. Aggregating Sentinel-2 pixels to match the lower spatial resolution grid of Landsat 8 using spatial reduction  and reprojection.
3. Integrating both aligned datasets into an interactive split-panel map  for side-by-side spatial inspection.



In [14]:
# Filter Landsat 8 collection for the same area and timeframe
# Filter Landsat 8 collection (sin aplicar el enmascaramiento de S2)
l8_collection = (
    ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
    .filterBounds(ee_muni_geom)
    .filterDate("2023-01-01", "2023-12-31")
    .filter(ee.Filter.lt("CLOUD_COVER", 30))
)

# Scaling function for Landsat 8 Surface Reflectance
#To apply a scale factor (0.0000275) and an offset (-0.2) to the Landsat 8 Level-2 Surface Reflectance bands (SR_B.)
def scale_landsat8(image):
    optical_bands = image.select('SR_B.').multiply(0.0000275).add(-0.2) 
    return image.addBands(optical_bands, None, True)

# Compute median mosaic, scale values, compute NDVI (B5 = NIR, B4 = Red), and clip
l8_scaled = l8_collection.map(scale_landsat8).median()
l8_ndvi_30m = l8_scaled.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI_L8').clip(ee_muni_geom)

print("Landsat 8 OLI NDVI successfully processed.")

Landsat 8 OLI NDVI successfully processed.


> 🎯 **Your Turn 8:** Look closely at the `scale_landsat8` function above. Why is it necessary to apply a scale factor (`0.0000275`) and an offset (`-0.2`) to the Landsat 8 Level-2 Surface Reflectance bands (`SR_B.`) before calculating spectral indices like NDVI?
>
> R=/ Landsat stores reflectance as scaled digital values. The scale factor and offset convert these values into actual surface reflectance, which allows us to calculate NDVI correctly.

### 4.1. Spatial Resampling and Coordinate Grid Alignment

This subsection handles the spatial alignment between datasets captured at different spatial resolutions.

While Sentinel-2 provides a native resolution of 10 metres per pixel, Landsat 8 operates at a coarser 30-metre resolution. To perform a valid pixel-by-pixel cross-sensor comparison, the 10-metre Sentinel-2 NDVI raster must be aggregated to match the 30-metre target spatial grid of Landsat 8.

1. Retrieves the native projection metadata (`ee.Projection`) from raw single-date collection assets prior to temporal reduction operations.
2. Aggregates higher-resolution (10m) pixels falling within each target lower-resolution (30m) pixel using a spatial mean reducer (`ee.Reducer.mean()`).
3.  Forces the resampled Sentinel-2 layer to adopt the exact Coordinate Reference System and pixel grid alignment of the target Landsat 8 raster .

---

In [15]:
# 1. Extract valid native projections from raw image items (before computing median)
s2_native_proj = s2_collection.first().select("B4").projection()
landsat_proj = l8_collection.first().select("SR_B5").projection()

# 2. Resample Sentinel-2 NDVI to match Landsat 8 grid using native resolution
s2_ndvi_30m = (
    ndvi_image
    .setDefaultProjection(crs=s2_native_proj)  
    .reduceResolution(
        reducer=ee.Reducer.mean(),
        maxPixels=1024
    )
    .reproject(
        crs=landsat_proj  # Align directly to Landsat 8 spatial reference frame
    )
    .clip(ee_muni_geom)
)

print("Sentinel-2 projection reference successfully assigned.")

Sentinel-2 projection reference successfully assigned.


> 🎯 **Your Turn 9:** Examine the spatial resampling chain applied to the Sentinel-2 NDVI layer (`.reduceResolution()` combined with `.reproject()`). 
> 
> Why is `ee.Reducer.mean()` used to aggregate the 10-metre Sentinel-2 pixels into the 30-metre Landsat grid instead of the resampling method like Nearest-Neighbour ?
>
> R=/ ee.Reducer.mean() in .reduceResolution() averages the 10-m Sentinel-2 pixels within each 30-m Landsat pixel, while .reproject() aligns the result to the Landsat grid. This preserves more information than nearest-neighbour resampling, which uses only the nearest pixel.


### 4.2. Comparative Visualisation using Split-Panel Interactive Maps

Sets up a side-by-side spatial comparison of the resampled Sentinel-2 NDVI * and the native Landsat 8 NDVI  layers over the municipal territory.

1. Converts Earth Engine raster objects into interactive map tiles using uniform visual parameters.
2. Instantiates a interactive map centered on the municipal centroid.
3. Binds both layers into a split-panel view , introducing an interactive slider handle to dynamically evaluate local spectral consistency across both sensors.


In [17]:
# # Install geemap to create interactive maps with Google Earth Engine
#%pip install geemap

import geemap

# Define standard NDVI visualization parameters
ndvi_vis = {
    'min': 0.0,
    'max': 0.7,
    'palette': ['blue', 'white', 'green']
}

# Convert Earth Engine images into TileLayers for geemap
left_layer = geemap.ee_tile_layer(s2_ndvi_30m, ndvi_vis, 'Sentinel-2 NDVI (30m)')
right_layer = geemap.ee_tile_layer(l8_ndvi_30m, ndvi_vis, 'Landsat 8 NDVI (30m)')

# Initialise split-panel map centred on the municipality
Split_Map = geemap.Map(center=centroid, zoom=11)
Split_Map.split_map(left_layer=left_layer, right_layer=right_layer)

# Display interactive split map
Split_Map

Map(center=[5.333397717843284, -74.40807977180083], controls=(ZoomControl(options=['position', 'zoom_in_text',…

**Interactive map with split panels:** The map allows for a comparison of the results obtained from the Landsat 8 OLI and the Sentinel-2 MSI following spatial resampling, based on a comparable spatial reference. In the case of the study area, the local spectral consistency between the two sensors shows similarities in the same sectors on both maps. Differences in color intensity and distribution may be due to the sensors’ different spectral characteristics and spatial resolutions.

## 5. Vector Masking & Spatial Extraction
Evolves from a single-year analysis into a multi-temporal baseline study using the Sentinel-2 Harmonised collection across a 10-year span (2016–2025).

1. Iterates over a sequence of years using  to build a custom  where each image represents a cloud-masked median NDVI composite for that specific year.
2. Enables server-side temporal filtering.
3. Calculates the pixel-wise mean  across the entire multi-year collection to establish the long-term historical average.
4. Derives the relative vegetation anomaly for a target year 2023 by subtracting the multi-year mean from the target year's composite:

$$\text{NDVI Anomaly}_{2023} = \text{NDVI}_{2023} - \text{NDVI}_{\text{Historical Baseline}}$$

> 💡Positive values indicate above-average vegetation vigour (greening), while negative values highlight vegetation stress, deforestation, or drought conditions relative to the long-term mean.

In [21]:
import geemap

# 1. Define range of available years for Sentinel-2 Harmonised (2016 to 2025)
years = ee.List.sequence(2017, 2025) #There are no images from 2016

# 2. Map function to generate an annual composite and compute NDVI per year
def compute_annual_ndvi(year):
    date_start = ee.Date.fromYMD(year, 1, 1)
    date_end = ee.Date.fromYMD(year, 12, 31)
    
    annual_s2 = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(ee_muni_geom)
        .filterDate(date_start, date_end)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 30))
        .map(mask_s2_clouds_scl)
        .median()
    )
    
    annual_ndvi = (
        annual_s2
        .normalizedDifference(["B8", "B4"])
        .rename("NDVI")
        .set("year", year)
    )
    
    return annual_ndvi

# Build ImageCollection containing one NDVI image per year
annual_ndvi_collection = ee.ImageCollection(years.map(compute_annual_ndvi))

# 3. Compute the long-term baseline average across all available years
historical_mean_ndvi = annual_ndvi_collection.mean().clip(ee_muni_geom)

# 4. Compute annual anomaly for a target year (e.g., 2023)
target_year = 2023
target_ndvi = (
    annual_ndvi_collection
    .filter(ee.Filter.eq("year", target_year))
    .first()
    .clip(ee_muni_geom)
)

# Difference calculation: Current Year minus Historical Baseline
ndvi_anomaly_2023 = target_ndvi.subtract(historical_mean_ndvi).rename("NDVI_Anomaly")

print(f"Annual NDVI Anomaly successfully computed for year {target_year}.")

Annual NDVI Anomaly successfully computed for year 2023.


In [18]:
#Check which images are available for each year
for year in range(2016, 2026): #There are no images from 2016
    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(ee_muni_geom)
        .filterDate(f"{year}-01-01", f"{year}-12-31")
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 30))
    )
    print(year, collection.size().getInfo())

2016 0
2017 3
2018 7
2019 26
2020 32
2021 22
2022 7
2023 14
2024 21
2025 15


> 🎯 **Your Turn 10:** Now that you have computed `ndvi_anomaly_2023`, write the Python code to display this anomaly layer on an interactive `geemap.Map`.
> 
> **Requirements:**
> 1. Define appropriate visualization parameters suited for differences:
>    - Set the range between `-0.3` (negative anomaly / vegetation degradation) and `0.3` (positive anomaly / vegetation growth).
>    - Use a diverging color palette such as `['red', 'white', 'green']` or `['brown', 'yellow', 'darkgreen']`.
> 2. Initialize a `geemap.Map` centered on the municipal centroid (`centroid`) with zoom level `11`.
> 3. Add `ndvi_anomaly_2023` to the map interface and display it.

In [23]:
# Write your script here:
anomaly_vis = {
    'min': -0.3,
    'max': 0.3,
    'palette': ['red', 'white', 'green']
}

Map_anomaly = geemap.Map(center=centroid, zoom=11)
Map_anomaly.add_layer(ndvi_anomaly_2023, anomaly_vis, 'NDVI Anomaly 2023')
Map_anomaly

Map(center=[5.333397717843284, -74.40807977180083], controls=(WidgetControl(options=['position', 'transparent_…

**NDVI Anomaly Map:** The results shows that during the year analyzed (2023), vegetation in the study area exhibited spatial variations from its historical pattern, with a predominance of mild negative anomalies. Light red tones predominate, indicating a relative reduction in vegetation vigor, possibly associated with plant stress, deforestation, or drought. White areas are also observed, corresponding to conditions close to the historical average, as well as small green areas representing above-average vigor.

## 5. Interactive Multi-Decadal Anomaly Visualisation 

Renders the complete time series of annual NDVI anomalies (2016–2025) onto a single interactive `geemap` canvas for temporal exploration.

1.  Defines a symmetrical ColorBrewer palette to distinguish vegetation gain from loss.
2. Adds the 10-year historical mean NDVI as an optional reference layer (hidden by default).
3. Loops through every year from 2016 to 2025, computing each year's departure from the historical baseline and adding it as a toggleable map layer (keeping 2023 active by default).
4. Constructs a custom dictionary mapping quantitative anomaly intervals to qualitative health states 
5. Appends an interactive legend box and layer control panel to enable rapid multi-temporal toggling and comparative analysis across years.


In [27]:
import geemap

# 1. Define explicit color palette: Red (Negative/Stress) -> Yellow (Neutral) -> Green (Positive/Healthy)
anomaly_vis = {
    'min': -0.3,
    'max': 0.3,
    'palette': ['#d73027', '#f46d43', '#fdae61', '#fee08b', '#ffffbf', '#d9ef8b', '#a6d96a', '#66bd63', '#1a9850']
}

# 2. Initialise interactive map centred on San Luis de Gaceno
Map_Anomalies = geemap.Map(center=centroid, zoom=11)

# 3. Add historical baseline mean layer (10-year reference, off by default)
Map_Anomalies.addLayer(
    historical_mean_ndvi, # Median selection 
    {'min': 0.0, 'max': 0.8, 'palette': ['blue', 'white', 'green']}, 
    'Baseline: Historical Mean NDVI (2016-2025)',
    False
)

# 4. Iteratively compute and add individual annual anomaly layers (2016-2025)
year_list = [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025] #There are no images from 2016

for y in year_list:
    # Extract annual NDVI layer for year 'y'
    annual_ndvi = (
        annual_ndvi_collection
        .filter(ee.Filter.eq("year", y))
         
        .first()
            
        .clip(ee_muni_geom)
    )
    
    # Calculate anomaly: (Year NDVI - Historical Baseline Mean)
    annual_anomaly = annual_ndvi.subtract(historical_mean_ndvi).rename(f"Anomaly_{y}")
    
    # Keep 2023 visible by default for immediate visual assessment
    show_layer = True if y == 2023 else False
    
    # Add annual anomaly layer with toggles
    Map_Anomalies.addLayer(
        annual_anomaly, 
        anomaly_vis, 
        f'NDVI Anomaly {y}', 
        show_layer
    )

# 5. Define discrete legend entries for student interpretation
legend_dict = {
    'Positive Anomaly (> +0.15 NDVI)': '#1a9850',  # Dark green
    'Slightly Positive (+0.05 to +0.15)': '#a6d96a', # Light green
    'Normal Baseline (-0.05 to +0.05)': '#ffffbf',  # Yellow/Neutral
    'Slightly Negative (-0.15 to -0.05)': '#fdae61', # Orange
    'Negative Anomaly (< -0.15 NDVI)': '#d73027'   # Red
}

# Add legend and layer control widgets to the map UI
Map_Anomalies.add_legend(title="NDVI Anomaly Legend", legend_dict=legend_dict)
Map_Anomalies.add_layer_control()

Map_Anomalies

Map(center=[5.333397717843284, -74.40807977180083], controls=(WidgetControl(options=['position', 'transparent_…

**2023 NDVI Anomaly Map:** Vegetation conditions close to their historical trend predominated throughout 2023, with a heterogeneous spatial distribution. Negative anomalies, represented mainly by shades of orange and red, indicate areas with lower vegetation vigor compared to the historical baseline. In contrast, some areas show positive anomalies, with shades of green, indicating higher vegetation vigor compared to historical trends.

### 5.1. Continuous Linear Visualisation of Annual NDVI Time Series

Renders the raw annual NDVI composites across the 10-year period (2016–2025) using a continuous sequential color ramp rather than anomaly differences.

1. Defines a continuous 5-step hex color gradient scaled linearly between `0.0` (bare soil/water) and `0.8` (dense vegetation canopy).
2. Integrates the multi-year baseline average (`historical_mean_ndvi`) as a reference layer for long-term canopy evaluation.
3. Filters `annual_ndvi_collection` year by year, clipping each annual composite to the municipal boundary and adding it to the map interface (setting 2023 visible by default).
4. Binds a horizontal colorbar widget irectly to the map canvas to represent absolute numerical vegetation density rather than discrete categories.
5. Adds layer controls to allow students to visually inspect inter-annual canopy variations across individual years.

---

In [29]:
import geemap

# 1. Define continuous linear visual parameters (pure white to deep forest green)
ndvi_vis_continuous = {
    'min': 0.0,
    'max': 0.8,
    'palette': ['#ffffff', '#e5f5e0', '#a1d99b', '#31a354', '#006d2c']
}

# 2. Initialise interactive map centred on La Palma
Map_Annual_NDVI_Linear = geemap.Map(center=centroid, zoom=11)

# 3. Add historical baseline mean layer as benchmark (off by default)
Map_Annual_NDVI_Linear.addLayer(
    historical_mean_ndvi, # Median selection 
    ndvi_vis_continuous, 
    'Baseline: Historical Mean NDVI (2016-2025)',
    False
)

# 4. Iteratively add individual annual NDVI layers (2016-2025)
year_list = [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025] #There are no images from 2016

for y in year_list:
    # Extract annual NDVI composite for year 'y'
    annual_ndvi = (
        annual_ndvi_collection
        .filter(ee.Filter.eq("year", y))
        .first()
        .clip(ee_muni_geom)
    )
    
    # Keep 2023 active by default; keep remaining years unchecked initially
    show_layer = True if y == 2023 else False
    
    # Add layer to interactive map
    Map_Annual_NDVI_Linear.addLayer(
        annual_ndvi, 
        ndvi_vis_continuous, 
        f'NDVI Year {y}', 
        show_layer
    )

# 5. Add a continuous linear colorbar legend (White to Green)
Map_Annual_NDVI_Linear.add_colorbar(
    vis_params=ndvi_vis_continuous,
    label="NDVI Index Value (Continuous)",
    orientation="horizontal",
    layer_name="NDVI Linear Scale"
)

# Add layer control toggle panel for students
Map_Annual_NDVI_Linear.add_layer_control()

# Display map
Map_Annual_NDVI_Linear

Map(center=[5.333397717843284, -74.40807977180083], controls=(WidgetControl(options=['position', 'transparent_…

### 5.2. Multi-Temporal True-Colour (RGB) Composite Exploration

Build a multi-year collection of natural-colour (True-Colour) imagery over the study area to provide visual ground truth across the 2016–2025 time horizon.
The aim is to have the natural-color image to facilitate the changes analysis over time. 
1. Constructs an image collection where each item is a cloud-masked, surface-reflectance median composite clipped to the municipal geometry (`ee_muni_geom`).
2. Applies standard Sentinel-2 RGB band mapping (`B4` = Red, `B3` = Green, `B2` = Blue) scaled between `0` and `3000` SR units. 
3.  Loops through each annual composite, loading every individual year onto the interactive map interface. Keeps 2023 active by default.


> 🎯 **Your Turn 11:** Review the custom `compute_annual_rgb` function defined above. Explain in your own words as a comment what each step inside the pipeline achieves:
> 
> 1. How does Earth Engine handle date boundaries when filtering image collections?
> 2. What is the functional difference between taking the median composite of an image collection versus taking a single image scene, and why is clipping performed at the very end of the pipeline?


In [32]:
def compute_annual_rgb(year):
    date_start = ee.Date.fromYMD(year, 1, 1)  # Define the first day of the year as the start date
    date_end = ee.Date.fromYMD(year, 12, 31) # Define December 31 as the end date; filterDate uses an exclusive end boundary
    
    annual_s2 = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(ee_muni_geom)
        .filterDate(date_start, date_end)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 30))
        .map(mask_s2_clouds_scl)
        .median() # Median combines multiple scenes into one representative composite, reducing the influence of outliers.
        .clip(ee_muni_geom) # Clip the final composite to the municipality after processing all available scenes.
    )
    
    return annual_s2.set("year", year)

# Build collection containing one S2 composite per year (2016-2025)
years = ee.List.sequence(2016, 2025)
annual_rgb_collection = ee.ImageCollection(years.map(compute_annual_rgb))

# 2. Define RGB visualization parameters (B4 = Red, B3 = Green, B2 = Blue)
rgb_vis = {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': 3000,
    'gamma': 1.4  # Slight gamma adjustment for optical clarity
}

# 3. Initialise interactive map centred on San Luis de Gaceno
Map_Annual_RGB = geemap.Map(center=centroid, zoom=11)

# 4. Iteratively add individual annual RGB layers (2016-2025)
year_list = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025] #There are no images from 2016

for y in year_list:
    # Filter the collection for year 'y'
    annual_rgb = (
        annual_rgb_collection
        .filter(ee.Filter.eq("year", y))
        .first()
    )
    
    # Keep 2023 visible by default for baseline verification
    show_layer = True if y == 2023 else False
    
    # Add annual RGB layer with individual checkboxes
    Map_Annual_RGB.addLayer(
        annual_rgb, 
        rgb_vis, 
        f'Sentinel-2 RGB {y}', 
        show_layer
    )

# Add layer control toggle panel for student verification
Map_Annual_RGB.add_layer_control()

# Display interactive map
Map_Annual_RGB


Map(center=[5.333397717843284, -74.40807977180083], controls=(WidgetControl(options=['position', 'transparent_…

## 6. Final Assessment & Independent Exercise: Multi-Sensor Alternative Spectral Index & Time-Series Export


🎯 **Your Turn 12 (Final Synthesis Challenge):**

> Create a new Jupyter Notebook in your GitHub repository and replicate what you  consider is necessary to reach waht is required as follows:
>
>  1. Choose one index distinct from NDVI (e.g., **EVI**, **SAVI**, or **NDWI**) for canopy, soil, or moisture evaluation.
> 2.Generate annual median composites for both **Sentinel-2** and **Landsat 8** over your area, applying sensor-specific cloud masking (SCL / QA_PIXEL) and Landsat 8 Surface Reflectance scaling factors.
> 3. **Construct Multi-Temporal Stacks:** Combine the 10 annual index layers (2016 to 2025) into a single 10-band `ee.Image` stack for Sentinel-2 and a corresponding 10-band stack for Landsat 8 (renaming bands sequentially as `Index_2016`, `Index_2017`, ..., `Index_2025`).
> 4. **GeoTIFF Export:** Export both 10-band image stacks to Google Drive as projected GeoTIFF rasters using the official local CRS for Colombia (**EPSG:9377**).

> 5. Commit and push your final `.ipynb` file to your public GitHub repository.
> 6. Ensure your notebook contains executed output cells, structured Markdown headers, and code comments explaining your custom study area selection.
> 7. Copy the direct link to your Jupyter Notebook on GitHub and submit it using the following link:
>
> 📋 [**Submit Your Final Notebook Link Here**](https://forms.gle/73ypriddjvWtarV8A)

---


**1. Enhanced Vegetation Index (EVI) and Annual Mean Composites (Sentinel-2 and Landsat 8)**

The EVI (Enhanced Vegetation Index) is a numerical indicator that measures the amount and health of green vegetation by analyzing the light reflected by plants and captured by a satellite. It is an optimization of the NDVI that incorporates the blue band to subtract atmospheric noise (such as dust or smoke) and a background correction factor to eliminate interference from bright ground surfaces. Its main technical advantage is that it maintains a linear response and does not saturate in areas with extreme canopy densities, such as rainforests or dense forests.

For Sentinel-2, the EVI was calculated using the standard formulation:

$$ EVI=2.5\frac{NIR-RED}{NIR+6(RED)-7.5(BLUE)+1} $$

Sentinel-2 MSI bands:

NIR: B8
Red: B4
Blue: B2

For Landsat 8 OLI, the same EVI formulation was applied using the corresponding surface reflectance bands:

NIR: SR_B5
Red: SR_B4
Blue: SR_B2

For Landsat 8, the surface reflectance scaling factors were applied before calculating EVI.

In [33]:
# ============================================================
# SENTINEL-2: Annual EVI Composites 2016-2025
# ============================================================

# Use the 500-metre buffer created previously
gdf_buffered_4326 = gdf_buffered.to_crs(epsg=4326)

geojson_buffer = gdf_buffered_4326.geometry.iloc[0].__geo_interface__

ee_buffer_geom = ee.Geometry(geojson_buffer)

# EVI for years to process
year_list = [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025] # Years 2016 no valid images 

s2_evi_images = []

for year in year_list:

    start_date = ee.Date.fromYMD(year, 1, 1)
    end_date = ee.Date.fromYMD(year + 1, 1, 1)

    # Sentinel-2 collection for the year
    yearly_collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(ee_buffer_geom)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 30))
        .map(mask_s2_clouds_scl)
    )

    # Number of images
    n_images = yearly_collection.size().getInfo()

    print(f"{year}: {n_images} images")

    # Skip the year if no images remain
    if n_images == 0:
        print(f"Skipping {year}: no valid images.")
        continue

    # Annual median composite
    annual = yearly_collection.median()

    # EVI
    evi = annual.expression(
        "2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))",
        {
            "NIR": annual.select("B8"),
            "RED": annual.select("B4"),
            "BLUE": annual.select("B2")
        }
    ).rename("EVI")

    # Clip EVI to the previously created 500-metre buffer
    evi = evi.clip(ee_buffer_geom)

    # Store metadata
    evi = evi.set("year", year)
    evi = evi.set("image_count", n_images)

    s2_evi_images.append(evi)


# Convert list to ImageCollection
s2_evi_collection = ee.ImageCollection.fromImages(s2_evi_images)

print("Sentinel-2 annual EVI collection created.")
print("Number of annual layers:", s2_evi_collection.size().getInfo())

2017: 3 images
2018: 7 images
2019: 26 images
2020: 32 images
2021: 22 images
2022: 7 images
2023: 14 images
2024: 21 images
2025: 15 images
Sentinel-2 annual EVI collection created.
Number of annual layers: 9


In [34]:
# ============================================================
# SENTINEL-2: Create 10-Band Multi-Temporal EVI Stack - Note: Year 2016 no valid images 
# ============================================================

all_years = [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025] # Year 2016 no valid images 

s2_evi_stack = None

for year in all_years:

    band_name = f"Index_{year}"

    # Check whether an EVI image exists for this year
    annual_evi = (
        s2_evi_collection
        .filter(ee.Filter.eq("year", year))
        .first()
    )

    # Years without Sentinel-2 data
    if year not in year_list:

        annual_evi = (
            ee.Image.constant(0)
            .rename(band_name)
            .updateMask(ee.Image.constant(0))
            .clip(ee_buffer_geom)
        )

    else:

        annual_evi = annual_evi.rename(band_name)

    # Build the multi-band stack
    if s2_evi_stack is None:
        s2_evi_stack = annual_evi
    else:
        s2_evi_stack = s2_evi_stack.addBands(annual_evi)

# Check results
print("Sentinel-2 EVI stack created.")
print("Number of bands:", s2_evi_stack.bandNames().size().getInfo())
print("Band names:", s2_evi_stack.bandNames().getInfo())

Sentinel-2 EVI stack created.
Number of bands: 9
Band names: ['Index_2017', 'Index_2018', 'Index_2019', 'Index_2020', 'Index_2021', 'Index_2022', 'Index_2023', 'Index_2024', 'Index_2025']


In [35]:
# Visualización del stack EVI de Sentinel-2

evi_vis = {
    "min": -0.2,
    "max": 1.0,
    "palette": ["red", "yellow", "green"]
}

Map_S2_EVI = geemap.Map(
    center=centroid,
    zoom=11
)

# Display EVI 2023
Map_S2_EVI.addLayer(
    s2_evi_stack.select("Index_2023"), #Change the year for a different view
    evi_vis,
    "Sentinel-2 EVI 2023" #Change the year for a different view
)

Map_S2_EVI

Map(center=[5.333397717843284, -74.40807977180083], controls=(WidgetControl(options=['position', 'transparent_…

In [37]:
# ============================================================
# LANDSAT 8: QA_PIXEL Cluud Mask
# ============================================================

def mask_landsat8_qa(image):

    qa = image.select("QA_PIXEL")

    mask = (
        qa.bitwiseAnd(1 << 0).eq(0)  # Fill
        .And(qa.bitwiseAnd(1 << 1).eq(0))  # Dilated cloud
        .And(qa.bitwiseAnd(1 << 2).eq(0))  # Cirrus
        .And(qa.bitwiseAnd(1 << 3).eq(0))  # Cloud
        .And(qa.bitwiseAnd(1 << 4).eq(0))  # Cloud shadow
    )

    return image.updateMask(mask)

In [38]:
# ============================================================
# LANDSAT 8: Surface Reflectance Scaling
# ============================================================
# Convert Landsat 8 SR bands from Digital Number to surface reflectance using the scale factor (0.0000275) and offset (-0.2).

def scale_landsat8(image):

    optical_bands = image.select("SR_B.*").multiply(0.0000275).add(-0.2)

    return image.addBands(
        optical_bands,
        overwrite=True
    )

In [39]:
# ============================================================
# LANDSAT 8: Annual EVI Composites 2016-2025
# ============================================================

# Use the 500-metre buffer created previously
gdf_buffered_4326 = gdf_buffered.to_crs(epsg=4326)

geojson_buffer = (
    gdf_buffered_4326
    .geometry
    .iloc[0]
    .__geo_interface__
)

ee_buffer_geom = ee.Geometry(geojson_buffer)


# Years to process
landsat_year_list = list(range(2016, 2026))

landsat_evi_images = []

for year in landsat_year_list:

    start_date = ee.Date.fromYMD(year, 1, 1)
    end_date = ee.Date.fromYMD(year + 1, 1, 1)

    yearly_collection = (
        ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
        .filterBounds(ee_buffer_geom)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt("CLOUD_COVER", 30))
        .map(mask_landsat8_qa)
        .map(scale_landsat8)
    )

    n_images = yearly_collection.size().getInfo()

    print(f"{year}: {n_images} images")

    if n_images == 0:

        print(f"Skipping {year}: no valid images.")
        continue

    # Annual median composite
    annual = yearly_collection.median()

    # EVI
    evi = annual.expression(
        "2.5 * ((NIR - RED) / "
        "(NIR + 6 * RED - 7.5 * BLUE + 1))",
        {
            "NIR": annual.select("SR_B5"),
            "RED": annual.select("SR_B4"),
            "BLUE": annual.select("SR_B2")
        }
    ).rename("EVI")

    # Clip EVI to the previously created 500-metre buffer
    evi = evi.clip(ee_buffer_geom)

    # Metadata
    evi = evi.set("year", year)
    evi = evi.set("image_count", n_images)

    landsat_evi_images.append(evi)


# Convert list to ImageCollection
landsat_evi_collection = ee.ImageCollection.fromImages(
    landsat_evi_images
)

print("Landsat 8 annual EVI collection created.")

print(
    "Number of annual layers:",
    landsat_evi_collection.size().getInfo()
)

2016: 1 images
2017: 4 images
2018: 6 images
2019: 4 images
2020: 10 images
2021: 6 images
2022: 1 images
2023: 5 images
2024: 3 images
2025: 3 images
Landsat 8 annual EVI collection created.
Number of annual layers: 10


In [40]:
# ============================================================
# LANDSAT 8: Create 10-Band Multi-Temporal EVI Stack
# ============================================================

all_years = list(range(2016, 2026))

landsat_evi_stack = None

for year in all_years:

    band_name = f"Index_{year}"

    annual_evi = (
        landsat_evi_collection
        .filter(ee.Filter.eq("year", year))
        .first()
        .rename(band_name)
    )

    if landsat_evi_stack is None:
        landsat_evi_stack = annual_evi
    else:
        landsat_evi_stack = landsat_evi_stack.addBands(
            annual_evi
        )

print("Landsat 8 EVI stack created.")

print(
    "Number of bands:",
    landsat_evi_stack.bandNames().size().getInfo()
)

print(
    "Band names:",
    landsat_evi_stack.bandNames().getInfo()
)

Landsat 8 EVI stack created.
Number of bands: 10
Band names: ['Index_2016', 'Index_2017', 'Index_2018', 'Index_2019', 'Index_2020', 'Index_2021', 'Index_2022', 'Index_2023', 'Index_2024', 'Index_2025']


In [41]:
# ============================================================
# LANDSAT 8 EVI 2023 Visualization
# ============================================================

evi_vis = {
    "min": -0.2,
    "max": 1.0,
    "palette": ["brown", "yellow", "green"] 
}

Map_Landsat_EVI = geemap.Map(
    center=centroid,
    zoom=11
)

# Display EVI 2023 clipped to the 500-metre buffer
Map_Landsat_EVI.addLayer(
    landsat_evi_stack.select("Index_2023"), #Change the year for a different view
    evi_vis,
    "Landsat 8 EVI 2023" #Change the year for a different view
)

Map_Landsat_EVI

Map(center=[5.333397717843284, -74.40807977180083], controls=(WidgetControl(options=['position', 'transparent_…

**GeoTIFF Export Sentinel-2: 7 Annual EVI collection created / Landsat 8: 8 Annual EVI collection created**
 

In [58]:
# ============================================================
# EXPORT EVI STACKS AS GeoTIFF
# Initial export CRS: EPSG:4326
# Final CRS will be EPSG:9377 after local reprojection
# ============================================================

# ------------------------------------------------------------
# Sentinel-2 EVI Stack
# ------------------------------------------------------------

sentinel_export = ee.batch.Export.image.toDrive(
    image=s2_evi_stack,
    description="Sentinel2_EVI_Stack_WGS84",
    folder="GEE_EVI_Exports",
    fileNamePrefix="Sentinel2_EVI_Stack_WGS84",
    region=ee_buffer_geom,
    scale=10,
    crs="EPSG:4326",
    fileFormat="GeoTIFF",
    maxPixels=1e13
)

sentinel_export.start()


# ------------------------------------------------------------
# Landsat 8 EVI Stack
# ------------------------------------------------------------

landsat_export = ee.batch.Export.image.toDrive(
    image=landsat_evi_stack,
    description="Landsat8_EVI_Stack_WGS84",
    folder="GEE_EVI_Exports",
    fileNamePrefix="Landsat8_EVI_Stack_WGS84",
    region=ee_buffer_geom,
    scale=30,
    crs="EPSG:4326",
    fileFormat="GeoTIFF",
    maxPixels=1e13
)

landsat_export.start()


# Set the spatial resolution: 10 m for Sentinel-2 and 30 m for Landsat 8.
print("Export tasks started.")
print("Sentinel-2: 10 m, EPSG:4326")
print("Landsat 8: 30 m, EPSG:4326")
print("Destination: Google Drive / GEE_EVI_Exports")

Export tasks started.
Sentinel-2: 10 m, EPSG:4326
Landsat 8: 30 m, EPSG:4326
Destination: Google Drive / GEE_EVI_Exports


Verification of the Process for Creating a Folder in Drive and Exporting as GeoTIFF Rasters

In [62]:
tasks = ee.batch.Task.list()

for task in tasks[:2]:
    status = task.status()
    print(
        f"Description: {status.get('description')}\n"
        f"State: {status.get('state')}\n"
        f"Error: {status.get('error_message')}\n"
        f"{'-' * 60}"
    )

Description: Landsat8_EVI_Stack_WGS84
State: FAILED
Error: Not enough space in Google Drive (need 23MB for this export).
------------------------------------------------------------
Description: Sentinel2_EVI_Stack_WGS84
State: FAILED
Error: Not enough space in Google Drive (need 188MB for this export).
------------------------------------------------------------
